In [ ]:
import copy
import os

import numpy as np
from matplotlib import pyplot as plt

from cascade.cascade import Cascade

datadir = "C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/"  # input datadir, add the default b3d files here
cascade = Cascade(
    datadir,
    name="default",
    elevation_file="barrier3d-default-elevation.npy",  # if using different elevation file, change this before "-elevation.npy"
    dune_file="barrier3d-default-dunes.npy",
    parameter_file="outwash-parameters.yaml",
    storm_file="barrier3d-default-storms.npy",  # same as "StormSeries_1kyrs_VCR_Berm1pt9m_Slope0pt04_01.npy"
    num_cores=1,  # cascade can run in parallel, can never specify more cores than that
    roadway_management_module=False,
    alongshore_transport_module=False,
    beach_nourishment_module=False,
    community_economics_module=False,
    outwash_module=True,
    alongshore_section_count=1,
    time_step_count=101,
    wave_height=1,  # ---------- for BRIE and Barrier3D --------------- #
    wave_period=7,
    wave_asymmetry=0.8,
    wave_angle_high_fraction=0.2,
    bay_depth=3.0,
    s_background=0.001,
    berm_elevation=1.9,
    MHW=0.46,
    beta=0.04,
    sea_level_rise_rate=0.004,
    sea_level_rise_constant=True,
    background_erosion=0.0,
    min_dune_growth_rate=0.25,
    max_dune_growth_rate=0.65,
    road_ele=1.7,  # ---------- roadway management --------------- #
    road_width=30,
    road_setback=30,
    dune_design_elevation=3.7,
    dune_minimum_elevation=2.2,
    trigger_dune_knockdown=False,
    group_roadway_abandonment=None,
    nourishment_interval=None,  # ---------- beach and dune ("community") management --------------- #
    nourishment_volume=300.0,
    overwash_filter=40,
    overwash_to_dune=10,
    number_of_communities=1,  # ---------- coastal real estate markets (in development) --------------- #
    sand_cost=10,
    taxratio_oceanfront=1,
    external_housing_market_value_oceanfront=6e5,
    external_housing_market_value_nonoceanfront=4e5,
    fixed_cost_beach_nourishment=2e6,
    fixed_cost_dune_nourishment=2e5,
    nourishment_cost_subsidy=10e6,
    house_footprint_x=15,
    house_footprint_y=20,
    beach_full_cross_shore=70,
    outwash_storms="outwash_1storm.npy",  # --------- outwasher (in development) ------------ #
    washout_to_shoreface=True,
)

In [ ]:
for time_step in range(cascade._nt - 1):
    # Print time step to screen (NOTE: time_index in each model is time_step+1)
    print("\r", "Time Step: ", time_step, end="")
    cascade.update()
    if cascade.b3d_break:
        break

    fig1 = plt.figure()
    ax1 = fig1.add_subplot(111)
    mat = ax1.matshow(
        np.flip(cascade.barrier3d[0]._DomainTS[time_step]) * 10,
        cmap="terrain",
        vmin=-3.0,
        vmax=3.0,
    )
    cbar = fig1.colorbar(mat)
    cbar.set_label("m MHW", rotation=270, labelpad=15)
    ax1.set_title("Initial Elevation")
    ax1.set_ylabel("barrier width (dam)")
    ax1.set_xlabel("barrier length (dam)")
    plt.gca().xaxis.tick_bottom()
    fig1.savefig(
        "D:/NC State/Outwasher/Output/cascade_domains/domain_"
        + str(time_step)
        + ".png",
        facecolor="w",
    )
    plt.close()

In [ ]:
import copy
import os

import numpy as np
from barrier3d import Barrier3d
from matplotlib import pyplot as plt

from cascade.outwasher import Outwasher

b3d2 = Barrier3d.from_yaml("C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/")
# years = np.load("C:/Users/Lexi/PycharmProjects/CASCADE/cascade/data/outwash_data/outwash_years.npy")

for t in range(1, 101):
    print("B3D time step: ", b3d2._time_index)

    fig1 = plt.figure()
    ax1 = fig1.add_subplot(111)
    mat = ax1.matshow(
        np.flip(b3d2.InteriorDomain) * 10,
        cmap="terrain",
        vmin=-3.0,
        vmax=3.0,
    )
    cbar = fig1.colorbar(mat)
    cbar.set_label("m MHW", rotation=270, labelpad=15)
    ax1.set_title("Initial Elevation")
    ax1.set_ylabel("barrier width (dam)")
    ax1.set_xlabel("barrier length (dam)")
    plt.gca().xaxis.tick_bottom()
    #     plt.show()
    fig1.savefig(
        "D:/NC State/Outwasher/Output/master_b3d_domains/domain_"
        + str(b3d2._time_index)
        + ".png",
        facecolor="w",
    )
    plt.close()

    b3d2.update()
    b3d2.update_dune_domain()